# Pertemuan 12: Asosiasi Data & Sistem Rekomendasi Dasar
**Nama Lengkap:** Roland Albertian Sehapikang  
**NIM:** 240401010249  
**Kelas:** IF403  
**Mata Kuliah:** Data Science  
**Program Studi:** S1 - PJJ Informatika, Universitas Siber Asia

In [15]:
import warnings

warnings.simplefilter("ignore")
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
          'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']


transaksi = []
for i in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(
        [str(x) for x in np.random.choice(produk, n_item, replace=False)]
    )


for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))

Contoh transaksi: [['Keju', 'Roti', 'Mentega', 'Kopi', 'Selai'], ['Roti', 'Kopi', 'Teh', 'Selai', 'Mentega'], ['Kopi', 'Susu', 'Teh']]
Jumlah transaksi: 50


In [17]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)

print("5 Baris Pertama Matriks Transaksi (One-Hot):")
print(df.head())

5 Baris Pertama Matriks Transaksi (One-Hot):
    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


In [18]:
from mlxtend.frequent_patterns import apriori

for ms in [0.05, 0.1, 0.2]:
    freq = apriori(df, min_support=ms, use_colnames=True)
    print(f'min_support={ms}: {len(freq)} itemset ditemukan')

freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)

print("\nTop 10 Frequent Itemset:")
print(freq_items.head(10))

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan

Top 10 Frequent Itemset:
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Selai, Teh)


In [19]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(freq_items, metric='confidence', min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)
rules.reset_index(drop=True, inplace=True)

print("Aturan Asosiasi Hasil Filtrasi (Top 10 berdasarkan Lift):")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

Aturan Asosiasi Hasil Filtrasi (Top 10 berdasarkan Lift):
        antecedents consequents  support  confidence      lift
0       (Keju, Teh)     (Telur)     0.12    0.857143  2.380952
1  (Mentega, Selai)      (Kopi)     0.10    0.625000  1.953125
2      (Roti, Gula)     (Selai)     0.10    1.000000  1.923077
3          (Sereal)   (Mentega)     0.14    0.777778  1.851852
4      (Teh, Telur)      (Keju)     0.12    0.600000  1.764706
5     (Selai, Kopi)   (Mentega)     0.10    0.714286  1.700680
6     (Keju, Telur)       (Teh)     0.12    0.750000  1.630435
7     (Selai, Gula)      (Roti)     0.10    0.500000  1.562500
8   (Mentega, Kopi)     (Selai)     0.10    0.714286  1.373626
9            (Roti)     (Selai)     0.22    0.687500  1.322115


**1.**   **Aturan mana yang paling kuat (Lift tertinggi)?**  
Aturan terkuat dipimpin oleh kombinasi **{Selai}** $\rightarrow$ **{Roti}** atau **{Roti}** $\rightarrow$ **{Selai}** (bergantung pada hasil randomisasi seed tepatnya, namun secara konsisten pola Roti dan Selai mendominasi).  

**2.**   **Apakah masuk akal secara bisnis?**  
Ya, sangat masuk akal secara bisnis karena produk tersebut merupakan barang komplementer (saling melengkapi) yang biasa dikonsumsi bersamaan untuk sarapan.





In [20]:
from sklearn.metrics.pairwise import cosine_similarity

katalog = pd.DataFrame({
    'produk': produk,
    'kategori': ['Bakery', 'Bakery', 'Dairy', 'Bakery', 'Dairy', 'Dairy', 'Minuman', 'Bumbu', 'Minuman', 'Dairy']
})

fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

def rekomendasi_serupa(nama_produk, top_n=3):
    idx = katalog.index[katalog['produk'] == nama_produk][0]

    skor = list(enumerate(sim_matrix[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)

    skor = [
        (i, sim)
        for i, sim in skor
        if i != idx and sim > 0
    ]

    return katalog.iloc[
        [i for i, _ in skor[:top_n]]
    ]['produk'].tolist()

print(rekomendasi_serupa('Roti'))

['Selai', 'Sereal']


In [21]:
produk_target = 'Roti'

rules_terkait = (
    rules[rules['antecedents'].apply(lambda x: produk_target in x)]
    .sort_values('lift', ascending=False)
)

print('=== Rekomendasi dari Association Rules ===')
print(rules_terkait[['consequents', 'confidence', 'lift']].head(5))

print('\n=== Rekomendasi dari Content-Based ===')
print(rekomendasi_serupa(produk_target))

=== Rekomendasi dari Association Rules ===
  consequents  confidence      lift
2     (Selai)      1.0000  1.923077
9     (Selai)      0.6875  1.322115

=== Rekomendasi dari Content-Based ===
['Selai', 'Sereal']




**1.   Apakah kedua pendekatan memberi rekomendasi yang konsisten?**  
Tidak selalu sama secara item, tetapi memiliki karakteristik unik. Association Rules merekomendasikan **Selai** karena data transaksi riil membuktikan keduanya sering dibeli bersama (cross-commodity relationship). Sementara Content-Based Filtering merekomendasikan produk dalam satu kategori Bakery seperti yang ada pada daftar katalog.  
**2.   Kapan sebaiknya menggunakan salah satu atau digabungkan (Hybrid)?**  
*   **Association Rules:** Sangat cocok digunakan untuk penataan tata letak produk di toko fisik (layouting), pembuatan paket promosi (bundling), atau e-commerce skala dasar.
*   **Content-Based Filtering:** Tepat digunakan saat sistem menghadapi masalah cold start (pengguna baru yang belum memiliki transaksi), karena sistem hanya bergantung pada metadata produk.
*   **Hybrid System:** Merupakan pilihan terbaik untuk industri skala produksi modern (seperti e-commerce besar). Dengan menggabungkannya, sistem dapat menyajikan rekomendasi yang kaya ragam (serendipity) dari sisi transaksi kolektif, sekaligus menjaga relevansi fungsi dari sisi kemiripan deskripsi barang.







# Kesimpulan

---

## 1. Apa yang Dipelajari

* **Pra-pemrosesan Data Transaksi:** Mempelajari teknik transformasi data transaksi mentah berupa *list of items* menjadi matriks *One-Hot Encoding* (Boolean) menggunakan `TransactionEncoder`.
* **Algoritma Apriori & Frequent Itemsets:** Memahami mekanisme pencarian kombinasi produk yang sering dibeli bersamaan (*frequent itemsets*) dengan menguji dampak variasi ambang batas *minimum support*.
* **Aturan Asosiasi (Association Rules):** Mengisolasi dan menganalisis pola hubungan antar-produk menggunakan tiga metrik utama:
  * **Support:** Frekuensi kemunculan kombinasi produk dalam seluruh transaksi.
  * **Confidence:** Tingkat kepastian bahwa produk B dibeli jika produk A dibeli.
  * **Lift:** Metrik kekuatan asosiasi. Nilai $\text{Lift} > 1$ menandakan hubungan asosiasi positif (produk saling melengkapi).
* **Content-Based Filtering:** Membangun sistem rekomendasi sederhana berbasis kemiripan atribut/kategori produk menggunakan *Cosine Similarity*.
* **Strategi Kombinasi (Hybrid System):** Menganalisis perbedaan mendasar serta potensi penggabungan antara metode *Association Rules* (berbasis riwayat transaksi kolektif) dan *Content-Based Filtering* (berbasis metadata katalog).

---

## 2. Temuan Utama

* **Sensitivitas Ambang Batas Support:**
  * Semakin rendah nilai `min_support`, semakin banyak *itemset* yang ditemukan ($0.2 \to 13$ itemset, $0.1 \to 44$ itemset, $0.05 \to 74$ itemset).
* **Produk & Kombinasi Paling Dominan:**
  * **Selai** merupakan item individu paling populer dengan nilai *support* sebesar **0.52** (muncul di 52% total transaksi), disusul oleh **Teh** (**0.46**).
  * Kombinasi 2 item paling sering muncul adalah **{Teh, Selai}** dengan *support* **0.24**.
* **Aturan Asosiasi Terkuat:**
  * Aturan dengan nilai **Lift tertinggi** dipimpin oleh `{Teh, Keju} -> {Telur}` ($\text{Lift} = 2.38$, $\text{Confidence} = 85.7\%$).
  * Aturan `{Roti, Gula} -> {Selai}` menghasilkan nilai **Confidence sempurna (100%)** dengan $\text{Lift} = 1.92$, membuktikan hubungan produk komplementer sarapan yang sangat kuat.
  * Aturan dasar `{Roti} -> {Selai}` terbukti memiliki asosiasi positif kuat ($\text{Lift} = 1.32$, $\text{Confidence} = 68.75\%$).
* **Komparasi Hasil Rekomendasi untuk 'Roti':**
  * **Association Rules:** Merekomendasikan **Selai** berdasarkan data transaksi riil yang membuktikan keduanya sering dibeli bersama (*cross-category relationship*).
  * **Content-Based:** Merekomendasikan **Selai, Sereal, dan Susu** berdasarkan kemiripan atribut kategori (*Bakery* & *Dairy*).

---

## 3. Keterbatasan & Pertanyaan yang Muncul

* **Keterbatasan Dataset Sintetis & Komputasi:**
  * Dataset praktikum ini bersifat sintetis (50 transaksi, 10 produk). Pada skala *e-commerce* nyata dengan puluhan ribu item, algoritma Apriori rentan mengalami hambatan performa komputasi (*bottleneck*) akibat banyaknya kombinasi *candidate itemsets*.
* **Masalah Cold-Start pada Association Rules:**
  * *Association Rules* sangat bergantung pada riwayat transaksi historis. Jika ada produk baru yang belum pernah dibeli, algoritma ini tidak dapat memberikan rekomendasi (*item cold-start problem*).
* **Diversitas Rekomendasi Content-Based:**
  * Penggunaan fitur yang hanya terbatas pada satu kolom `kategori` membuat rekomendasi *Content-Based* berpotensi terlalu homogen (*low serendipity*).
* **Pertanyaan Lanjutan untuk Penggembalaan/Produksi:**
  * *Bagaimana formulasi terbaik untuk menggabungkan skor dari Association Rules (Lift) dan Content-Based (Cosine Score) ke dalam satu bobot terpadu pada Hybrid Recommender System di e-commerce nyata?*
  * *Apakah algoritma yang lebih modern seperti FP-Growth lebih efisien dibanding Apriori untuk menangani transaksi berukuran sangat besar?*